In [ ]:
import pandas as pd
import numpy as np
import sys
sys.path.append('..')
from src.hotel_booking_cancelation_prediction.feature_pipeline import load_processed_data

x_train_fs, x_test_fs, y_train, y_test = load_processed_data(output_dir='../data/processed')

print("x_train_fs:", x_train_fs.shape)
print("x_test_fs :", x_test_fs.shape)
print("y_train   :", y_train.shape)
print("y_test    :", y_test.shape)

# Critical check before trusting anything downstream
print("\nlead_time correlation:", x_train_fs['lead_time'].corr(y_train))
# Should be ~0.237

x_train_fs: (61060, 37)
x_test_fs : (26169, 37)
y_train   : (61060,)
y_test    : (26169,)

lead_time correlation: 0.23729770128766703


In [4]:
print("Shape:", x_train_fs.shape)
print("Non-numeric columns:", x_train_fs.select_dtypes(exclude=[np.number]).columns.tolist())
print("NaNs:", x_train_fs.isnull().sum().sum())
print("Infinite values:", np.isinf(x_train_fs.select_dtypes(include=[np.number])).sum().sum())
print("Sample values (should look scaled, roughly centered around 0):")
print(x_train_fs[['adr', 'lead_time']].describe())

Shape: (61060, 37)
Non-numeric columns: []
NaNs: 0
Infinite values: 0
Sample values (should look scaled, roughly centered around 0):
                adr     lead_time
count  6.106000e+04  6.106000e+04
mean  -1.536057e-17 -1.917744e-16
std    1.000008e+00  1.000008e+00
min   -5.710709e+00 -2.171343e+00
25%   -2.638303e-01 -6.442440e-01
50%    1.215592e-01  2.327900e-01
75%    5.135186e-01  8.007931e-01
max    1.556215e+00  1.529806e+00


In [5]:
import optuna
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def objective(trial):
    algo = trial.suggest_categorical('algorithm', ['logistic_regression', 'random_forest', 'xgboost', 'lightgbm'])

    if algo == 'logistic_regression':
        model = LogisticRegression(
            # C = inverse of regularization strength. Lower C = stronger regularization
            # (simpler model, less overfitting). Higher C = model fits training data more closely.
            C=trial.suggest_float('lr_C', 0.01, 10, log=True),
            max_iter=2000,
            random_state=42
        )

    elif algo == 'random_forest':
        model = RandomForestClassifier(
            # n_estimators = number of trees in the forest. More trees = more stable
            # predictions but slower training, with diminishing returns past a point.
            n_estimators=trial.suggest_int('rf_n_estimators', 100, 400),
            # max_depth = how deep each tree can grow. Deeper = more complex patterns
            # captured, but higher risk of overfitting to training data.
            max_depth=trial.suggest_int('rf_max_depth', 5, 25),
            # min_samples_leaf = minimum samples required in a leaf node. Higher = smoother,
            # more generalized decision boundaries (less overfitting).
            min_samples_leaf=trial.suggest_int('rf_min_samples_leaf', 1, 10),
            random_state=42,
            n_jobs=-1
        )

    elif algo == 'xgboost':
        model = XGBClassifier(
            # n_estimators = number of boosting rounds (sequential trees, each correcting
            # the previous one's errors).
            n_estimators=trial.suggest_int('xgb_n_estimators', 100, 400),
            # max_depth = depth of each individual tree. Boosting trees are usually shallower
            # than Random Forest trees since they build on each other sequentially.
            max_depth=trial.suggest_int('xgb_max_depth', 3, 10),
            # learning_rate = how much each tree's correction contributes to the final result.
            # Lower = slower but more careful learning, needs more trees to compensate.
            learning_rate=trial.suggest_float('xgb_learning_rate', 0.01, 0.3, log=True),
            # subsample = fraction of training rows used per tree. <1.0 adds randomness,
            # helping prevent overfitting (similar idea to bagging).
            subsample=trial.suggest_float('xgb_subsample', 0.6, 1.0),
            eval_metric='auc',
            random_state=42,
            n_jobs=-1
        )

    elif algo == 'lightgbm':
        model = LGBMClassifier(
            # Same meaning as XGBoost's n_estimators — number of sequential boosting trees.
            n_estimators=trial.suggest_int('lgbm_n_estimators', 100, 400),
            # num_leaves = max leaves per tree. LightGBM grows leaf-wise (not depth-wise),
            # so this controls model complexity more directly than max_depth would.
            num_leaves=trial.suggest_int('lgbm_num_leaves', 15, 100),
            # Same meaning as XGBoost's learning_rate.
            learning_rate=trial.suggest_float('lgbm_learning_rate', 0.01, 0.3, log=True),
            random_state=42,
            n_jobs=-1,
            verbose=-1
        )

    try:
        score = cross_val_score(
            model, x_train_fs, y_train,
            cv=cv, scoring='roc_auc', n_jobs=-1
        ).mean()
        return score
    except Exception as e:
        print(f"Trial failed ({algo}): {e}")
        return float('-inf')


study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler(seed=42))
study.optimize(objective, n_trials=50, show_progress_bar=True)

print("=" * 60)
print(f"WINNING ALGORITHM: {study.best_params['algorithm']}")
print(f"BEST PARAMS: {study.best_params}")
print(f"BEST CV ROC-AUC: {study.best_value:.4f}")
print("=" * 60)

c:\Users\lenovo\Desktop\HOTEL_BOOKING_CANCELATION_PREDICTION\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[I 2026-08-04 17:36:57,448] A new study created in memory with name: no-name-2fe3324d-fa31-47ab-bdcc-28c42e4cdeb4
Best trial: 0. Best value: 0.878505:   2%|▏         | 1/50 [00:33<27:21, 33.50s/it]

[I 2026-08-04 17:37:30,922] Trial 0 finished with value: 0.8785048119470911 and parameters: {'algorithm': 'random_forest', 'rf_n_estimators': 146, 'rf_max_depth': 8, 'rf_min_samples_leaf': 1}. Best is trial 0 with value: 0.8785048119470911.


Best trial: 0. Best value: 0.878505:   4%|▍         | 2/50 [00:47<17:23, 21.74s/it]

[I 2026-08-04 17:37:44,452] Trial 1 finished with value: 0.847351433312397 and parameters: {'algorithm': 'logistic_regression', 'lr_C': 8.123245085588687}. Best is trial 0 with value: 0.8785048119470911.


Best trial: 0. Best value: 0.878505:   6%|▌         | 3/50 [00:51<10:58, 14.00s/it]

...

[I 2026-08-04 18:05:36,940] Trial 47 finished with value: 0.9107589115215765 and parameters: {'algorithm': 'xgboost', 'xgb_n_estimators': 314, 'xgb_max_depth': 7, 'xgb_learning_rate': 0.12383378369676853, 'xgb_subsample': 0.7664440669266115}. Best is trial 31 with value: 0.9129888577816854.


Best trial: 31. Best value: 0.912989:  98%|█████████▊| 49/50 [29:27<00:37, 37.02s/it]

[I 2026-08-04 18:06:25,410] Trial 48 finished with value: 0.9123645163925392 and parameters: {'algorithm': 'xgboost', 'xgb_n_estimators': 355, 'xgb_max_depth': 9, 'xgb_learning_rate': 0.030435240927592814, 'xgb_subsample': 0.8303413754241195}. Best is trial 31 with value: 0.9129888577816854.


Best trial: 31. Best value: 0.912989: 100%|██████████| 50/50 [29:59<00:00, 35.99s/it]

[I 2026-08-04 18:06:56,756] Trial 49 finished with value: 0.9122326115106416 and parameters: {'algorithm': 'xgboost', 'xgb_n_estimators': 272, 'xgb_max_depth': 8, 'xgb_learning_rate': 0.059117937661977635, 'xgb_subsample': 0.7762528879971833}. Best is trial 31 with value: 0.9129888577816854.
WINNING ALGORITHM: xgboost
BEST PARAMS: {'algorithm': 'xgboost', 'xgb_n_estimators': 341, 'xgb_max_depth': 9, 'xgb_learning_rate': 0.04962533902377326, 'xgb_subsample': 0.8473942417329839}
BEST CV ROC-AUC: 0.9130


In [6]:
trials_df = study.trials_dataframe()
trials_df['algorithm'] = trials_df['params_algorithm']
trials_df_valid = trials_df[trials_df['value'] > 0]

summary = trials_df_valid.groupby('algorithm')['value'].agg(['max', 'mean', 'std', 'count']).sort_values('max', ascending=False)
print(summary)

                          max      mean       std  count
algorithm                                               
xgboost              0.912989  0.909774  0.004547     31
lightgbm             0.912304  0.902885  0.012678      6
random_forest        0.905090  0.889252  0.018834      6
logistic_regression  0.847360  0.847248  0.000176      7


In [ ]:
from xgboost import XGBClassifier

best_params = study.best_params

model = XGBClassifier(
    n_estimators=best_params['xgb_n_estimators'],
    max_depth=best_params['xgb_max_depth'],
    learning_rate=best_params['xgb_learning_rate'],
    subsample=best_params['xgb_subsample'],
    eval_metric='auc',
    random_state=42,
    n_jobs=-1
)

model.fit(x_train_fs, y_train)

y_pred = model.predict(x_test_fs)
y_pred_proba = model.predict_proba(x_test_fs)[:, 1] 
# predict_proba() actually returns two columns per row: [probability of class 0, probability of class 1] 
# (i.e. [P(not canceled), P(canceled)]). The [:, 1] just grabs the second column — the probability of cancellation specifically. 
# So y_pred_proba[0] = 0.83 means "the model thinks this booking has an 83% chance of being canceled."

In [8]:
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix

print("ROC-AUC:", roc_auc_score(y_test, y_pred_proba))

print("\nClassification Report:")
print(classification_report(y_test, y_pred, target_names=['Not Canceled', 'Canceled']))

print("\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred)
print(cm)
print(f"\nTrue Negatives : {cm[0,0]}  (correctly predicted NOT canceled)")
print(f"False Positives: {cm[0,1]}  (predicted canceled, actually didn't)")
print(f"False Negatives: {cm[1,0]}  (predicted not canceled, but actually did)")
print(f"True Positives : {cm[1,1]}  (correctly predicted canceled)")

ROC-AUC: 0.9140129664690277

Classification Report:
              precision    recall  f1-score   support

Not Canceled       0.88      0.92      0.90     18967
    Canceled       0.75      0.68      0.71      7202

    accuracy                           0.85     26169
   macro avg       0.82      0.80      0.81     26169
weighted avg       0.85      0.85      0.85     26169


Confusion Matrix:
[[17356  1611]
 [ 2310  4892]]

True Negatives : 17356  (correctly predicted NOT canceled)
False Positives: 1611  (predicted canceled, actually didn't)
False Negatives: 2310  (predicted not canceled, but actually did)
True Positives : 4892  (correctly predicted canceled)


In [13]:
import sys
sys.path.append('../src')
from src.hotel_booking_cancelation_prediction.training_model import train_xgboost
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score


# Train the model and get it back
model = train_xgboost(x_train_fs, y_train)

# Now you can evaluate or predict

# 1. Get predicted classes (0 for Not Canceled, 1 for Canceled)
y_pred = model.predict(x_test_fs)

# 2. Get predicted probabilities (needed for ROC-AUC score)
y_probs = model.predict_proba(x_test_fs)[:, 1]

🚀 Training winning XGBoost model...
✅ Training complete!


In [14]:
# Print ROC-AUC Score
roc_auc = roc_auc_score(y_test, y_probs)
print(f"ROC-AUC Score: {roc_auc:.4f}\n")

# Print Classification Report
print("--- Classification Report ---")
print(classification_report(y_test, y_pred))

# Print Confusion Matrix
print("--- Confusion Matrix ---")
print(confusion_matrix(y_test, y_pred))

ROC-AUC Score: 0.9140

--- Classification Report ---
              precision    recall  f1-score   support

           0       0.88      0.92      0.90     18967
           1       0.75      0.68      0.71      7202

    accuracy                           0.85     26169
   macro avg       0.82      0.80      0.81     26169
weighted avg       0.85      0.85      0.85     26169

--- Confusion Matrix ---
[[17356  1611]
 [ 2310  4892]]


In [16]:
import pandas as pd
from sklearn.model_selection import train_test_split
from src.hotel_booking_cancelation_prediction.cleaning import prepare_for_feature_pipeline
from src.hotel_booking_cancelation_prediction.preprocessor import HotelBookingPreprocessor

df = prepare_for_feature_pipeline(pd.read_csv("../data/raw/hotel_bookings.csv"))
X = df.drop(columns=["is_canceled"])
y = df["is_canceled"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=0, stratify=y
)

pre = HotelBookingPreprocessor()
X_train_ready = pre.fit_transform(X_train)
X_test_ready = pre.transform(X_test)

print(X_train_ready.shape)  # expect (61060, 37)
print(X_test_ready.shape)   # expect (26169, 37)

(61060, 37)
(26169, 37)


In [18]:
from src.hotel_booking_cancelation_prediction.predict import CancellationPredictor
import pandas as pd

raw = pd.read_csv("../data/raw/hotel_bookings.csv").head(20)
results = CancellationPredictor().predict(raw)
print(results.head())

   _original_row  cancel_probability  cancel_prediction risk_label
0              0            0.077075                  0   Low Risk
1              1            0.071641                  0   Low Risk
2              2            0.008336                  0   Low Risk
3              3            0.031225                  0   Low Risk
4              4            0.260192                  0   Low Risk
